# Visualization and Analysis of Satellite-Derived Chlorophyll Data

This notebook processes and visualizes chlorophyll-a concentration time series derived from multiple satellite platforms (Sentinel-2 and MODIS).

## Overview

- Purpose: Process, clean, and visualize chlorophyll/NDCI time series from satellite data
- Input: CSV files generated from Google Earth Engine processing
- Output: Time series plots of chlorophyll

## Key Processing Steps:

1. Data Import: Read CSV files with date, NDCI/chlorophyll values
2. NDCI Conversion: Convert NDCI to chlorophyll-a concentration (Sentinel)
3. Outlier Removal: Apply Median Absolute Deviation (MAD) method
4. Visualization: Generate time series plots for trend analysis

## Datasets:

- Sentinel-2: 10 m resolution, 5-day revisit, NDCI-based
- MODIS: 500 m resolution, daily, direct chlorophyll algorithm

In [ ]:
"""
Import required libraries for data processing and visualization.

Libraries:
- pandas: Data manipulation and CSV reading
- matplotlib: Plotting and visualization
- pathlib: File path handling
- numpy: Numerical operations and outlier detection
"""

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

In [ ]:
# ============================================================================
# UTILITY FUNCTIONS FOR DATA PROCESSING AND VISUALIZATION
# ============================================================================

def read_data(inpath):
    """
    Read satellite data from CSV file and prepare for analysis.
    
    Args:
        inpath: Path to CSV file containing satellite data
    
    Returns:
        DataFrame with parsed dates and sorted chronologically
    
    Processing:
    - Converts 'date' column to datetime format
    - Sorts data chronologically
    - Handles invalid date formats gracefully
    """
    df = pd.read_csv(inpath)
    # Parse dates with error handling for invalid formats
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df.sort_values('date', inplace=True)
    return df

def NDCI_to_chlorophyll(df, slope, offset):
    """
    Convert NDCI values to chlorophyll-a concentration.
    
    Args:
        df: DataFrame with 'ndci' column
        slope: Linear conversion factor (µg/L per NDCI unit)
        offset: Y-intercept of conversion equation
    
    Returns:
        DataFrame with added 'chl' column
    
    Algorithm:
        Chl-a = (NDCI x slope) + offset
    
    Note: This is an empirical conversion that may need calibration
    for specific water bodies.
    """
    df['chl'] = df['ndci'] * slope + offset
    return df

def remove_outliers(df, value_col='chl', threshold=5.0):
    """
    Remove outliers using Median Absolute Deviation (MAD) method.
    
    Args:
        df: Input DataFrame
        value_col: Column name for outlier detection (default: 'chl')
        threshold: Z-score threshold for outlier removal (default: 5.0)
    
    Returns:
        DataFrame with outliers removed
    
    Method:
    The MAD method is robust to outliers and works well for non-normal
    distributions common in environmental data:
    1. Calculate median of the data
    2. Calculate MAD = median(|x - median|)
    3. Convert MAD to robust standard deviation (sigma = 1.4826 x MAD)
    4. Remove points with |z-score| > threshold
    
    The factor 1.4826 makes MAD consistent with standard deviation
    for normally distributed data.
    """
    median = df[value_col].median()
    mad = np.median(np.abs(df[value_col] - median))
    
    # Handle case where all values are identical (MAD = 0)
    if mad == 0:
        return df.copy()
    
    # Convert MAD to robust standard deviation estimate
    robust_sd = 1.4826 * mad
    
    # Calculate z-scores and filter outliers
    z_scores = np.abs(df[value_col] - median) / robust_sd
    cleaned_df = df[z_scores < threshold].copy()
    
    return cleaned_df

def plot_data(df, label, title, ylabel, save_filename=None):
    """
    Create time series plot of chlorophyll data and optionally save to file.
    
    Args:
        df: DataFrame with 'date' and 'chl' columns
        label: Legend label for the data series
        title: Plot title
        ylabel: Y-axis label
        save_filename: If provided, save figure to this filename (PNG format)
    
    Returns:
        matplotlib axes object for further customization
    """
    # Create figure and axes
    fig, axes = plt.subplots(1, 1, figsize=(12, 6), sharex=True)
    
    # Define plot styling
    marker = ''           # No markers
    linestyle = '-'       # Solid line
    rgb = (60, 50, 255)   # Blue
    linecolor = '#{:02X}{:02X}{:02X}'.format(*rgb)
    
    # Plot the time series
    axes.plot(df['date'], df['chl'], 
              marker=marker, 
              linestyle=linestyle, 
              color=linecolor, 
              label=label)
    
    # Configure plot appearance
    axes.set_title(title)
    axes.set_ylabel(ylabel)
    axes.grid(True, alpha=0.3)
    
    # Adjust layout
    fig.tight_layout()
    
    # Save figure if filename is provided
    if save_filename:
        # Ensure filename ends with .png
        if not save_filename.endswith('.png'):
            save_filename += '.png'
        
        # Save at 300 DPI
        fig.savefig(save_filename, dpi=300, bbox_inches='tight')
        print(f"Figure saved as: {save_filename}")
    
    return axes

## Sentinel-2 Analysis - Detroit Lake

In [ ]:
"""
Process and visualize Sentinel-2 chlorophyll data for Detroit Lake.
"""

# Define input CSV filename
csv_filename = 'Detroit_S2_NDCI_500m.csv'
png_filename = csv_filename.replace('.csv', '.png')

# Load Sentinel-2 NDCI data from CSV
df = read_data(csv_filename)

# Convert NDCI to chlorophyll-a concentration
offset = 3    # Baseline chlorophyll level for oligotrophic lake (µg/L)
slope = 30    # Conservative conversion factor for clear water (µg/L per NDCI unit)
df = NDCI_to_chlorophyll(df, slope, offset)

# Optional: Remove outliers using MAD method
# df_clean = remove_outliers(df, 'chl', threshold=5.0)

# Generate time series plot and save to PNG
label = 'Sentinel-2 Chl-a'
title = 'Detroit Lake - Sentinel-2 Chlorophyll Time Series (2015-2025)'
ylabel = 'Chl-a (µg/L)'
axes = plot_data(df, label, title, ylabel, save_filename=png_filename)

## Sentinel-2 Analysis - Upper Klamath Lake

In [ ]:
"""
Process and visualize Sentinel-2 chlorophyll data for Upper Klamath Lake.
"""

# Define input CSV filename
csv_filename = 'Klamath_S2_NDCI_500m.csv'
png_filename = csv_filename.replace('.csv', '.png')

# Load Sentinel-2 NDCI data from CSV
df = read_data(csv_filename)

# Convert NDCI to chlorophyll-a concentration
offset = 3   # Baseline chlorophyll level for eutrophic lake (µg/L)
slope = 30   # Higher conversion factor for productive waters (µg/L per NDCI unit)
df = NDCI_to_chlorophyll(df, slope, offset)

# Optional: Remove outliers using MAD method
# df_clean = remove_outliers(df, 'chl', threshold=5.0)

# Generate time series plot and save to PNG
label = 'Sentinel-2 Chl-a'
title = 'Upper Klamath Lake - Sentinel-2 Chlorophyll Time Series (2015-2025)'
ylabel = 'Chl-a (µg/L)'
axes = plot_data(df, label, title, ylabel, save_filename=png_filename)

## MODIS Analysis - Detroit Lake

In [ ]:
"""
Process and visualize MODIS chlorophyll data for Detroit Lake.
"""

# Define input CSV and output PNG filenames
aqua_csv = 'Detroit_MODIS_Aqua_500m_Chl_singlePixel.csv'
terra_csv = 'Detroit_MODIS_Terra_500m_Chl_singlePixel.csv'
aqua_png = aqua_csv.replace('.csv', '.png')
terra_png = terra_csv.replace('.csv', '.png')

# Load MODIS data from both satellites
aqua_df = read_data(aqua_csv)
terra_df = read_data(terra_csv)

# Apply outlier removal - critical for MODIS due to:
# - Residual cloud effects
# - Atmospheric correction errors
# - Sun glint contamination
aqua_df_clean = remove_outliers(aqua_df, 'chl', threshold=5.0)
terra_df_clean = remove_outliers(terra_df, 'chl', threshold=5.0)

# Plot MODIS-Aqua (afternoon overpass ~1:30 PM local time) and save
label = 'MODIS-Aqua Chl-a'
title = 'Detroit Lake - MODIS-Aqua Chlorophyll Time Series (2011-2025)'
ylabel = 'Chl-a (µg/L)'
axes_aqua = plot_data(aqua_df_clean, label, title, ylabel, save_filename=aqua_png)

# Plot MODIS-Terra (morning overpass ~10:30 AM local time) and save
label = 'MODIS-Terra Chl-a'
title = 'Detroit Lake - MODIS-Terra Chlorophyll Time Series (2011-2025)'
ylabel = 'Chl-a (µg/L)'
axes_terra = plot_data(terra_df_clean, label, title, ylabel, save_filename=terra_png)

'''
Note: Differences between Terra and Aqua may exist due to:

- Diurnal variability in algae vertical distribution
- Different sun angle effects
- Sensor calibration differences
'''

## MODIS Analysis - Upper Klamath Lake

In [ ]:
"""
Process and visualize MODIS chlorophyll data for Upper Klamath Lake.
"""

# Define input CSV and output PNG filenames
aqua_csv = 'Klamath_MODIS_Aqua_500m_Chl_singlePixel.csv'
terra_csv = 'Klamath_MODIS_Terra_500m_Chl_singlePixel.csv'
aqua_png = aqua_csv.replace('.csv', '.png')
terra_png = terra_csv.replace('.csv', '.png')

# Load MODIS data from both satellites
aqua_df = read_data(aqua_csv)
terra_df = read_data(terra_csv)

# Apply outlier removal - critical for MODIS due to:
# - Residual cloud effects
# - Atmospheric correction errors
# - Sun glint contamination
aqua_df_clean = remove_outliers(aqua_df, 'chl', threshold=5.0)
terra_df_clean = remove_outliers(terra_df, 'chl', threshold=5.0)

# Plot MODIS-Aqua (afternoon overpass ~1:30 PM local time)
label = 'MODIS-Aqua Chl-a'
title = 'Klamath Lake - MODIS-Aqua Chlorophyll Time Series (2011-2025)'
ylabel = 'Chl-a (µg/L)'
axes_aqua = plot_data(aqua_df_clean, label, title, ylabel, save_filename=aqua_png)

# Plot MODIS-Terra (morning overpass ~10:30 AM local time)
label = 'MODIS-Terra Chl-a'
title = 'Klamath Lake - MODIS-Terra Chlorophyll Time Series (2011-2025)'
ylabel = 'Chl-a (µg/L)'
axes_terra = plot_data(terra_df_clean, label, title, ylabel, save_filename=terra_png)

'''
Note: Differences between Terra and Aqua may exist due to:

- Diurnal variability in algae vertical distribution
- Different sun angle effects
- Sensor calibration differences
'''

## Cross-Sensor Calibration

### Overview

This section performs cross-sensor calibration by comparing MODIS chlorophyll estimates with NDCI values from Sentinel-2. This helps derive lake-specific coefficients for NDCI-to-chlorophyll conversion.

### Method:

1. Match observations from different sensors by date (±1 day tolerance)
2. Use MODIS chlorophyll as reference truth
3. Fit regression models to derive optimal slope and offset
4. Compare linear vs. exponential relationships
5. Validate calibration with residual analysis

In [ ]:
"""
Cross-sensor calibration functions for deriving lake-specific 
NDCI-to-chlorophyll conversion coefficients.
"""

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

def match_observations(df1, df2, date_col='date', tolerance_days=1):
    """
    Match observations from two sensors based on date proximity.
    
    Args:
        df1: First dataframe (MODIS)
        df2: Second dataframe (Sentinel-2)
        date_col: Name of date column
        tolerance_days: Maximum days difference for matching
    
    Returns:
        Merged dataframe with matched observations
    """
    # Create copies to avoid modifying original data
    df1_copy = df1.copy()
    df2_copy = df2.copy()
    
    # Ensure date columns are datetime
    df1_copy[date_col] = pd.to_datetime(df1_copy[date_col])
    df2_copy[date_col] = pd.to_datetime(df2_copy[date_col])
    
    # Sort by date
    df1_copy = df1_copy.sort_values(date_col)
    df2_copy = df2_copy.sort_values(date_col)
    
    # Reset index to avoid issues
    df1_copy = df1_copy.reset_index(drop=True)
    df2_copy = df2_copy.reset_index(drop=True)
    
    # Use merge_asof on the datetime column directly
    matched = pd.merge_asof(
        df1_copy,
        df2_copy,
        on=date_col,
        direction='nearest',
        tolerance=pd.Timedelta(days=tolerance_days),
        suffixes=('_ref', '_cal')
    )
    
    # Remove rows where no match was found (NaN in columns from df2)
    # Check for NaN in the first non-date column from df2
    col_to_check = [col for col in matched.columns if col.endswith('_cal')][0]
    matched = matched.dropna(subset=[col_to_check])
    
    return matched

def exponential_model(x, a, b, c):
    """
    Exponential model for NDCI to chlorophyll conversion.
    Chl = a * exp(b * NDCI) + c
    """
    return a * np.exp(b * x) + c

def calibrate_ndci_to_chl(modis_df, ndci_df, lake_name="Lake"):
    """
    Calibrate NDCI to chlorophyll using MODIS as reference.
    
    Args:
        modis_df: MODIS chlorophyll dataframe
        ndci_df: Sentinel NDCI dataframe
        lake_name: Name of lake for plot titles
    
    Returns:
        Dictionary with calibration results
    """
    # Match observations
    matched = match_observations(modis_df, ndci_df)
    
    if len(matched) < 10:
        print(f"Warning: Only {len(matched)} matched observations found")
        if len(matched) < 3:
            print("Not enough matched observations for calibration")
            return None
    
    # Debug: print column names to verify
    print(f"Matched columns: {matched.columns.tolist()}")
    print(f"First few matched rows:\n{matched.head()}")
    
    # Extract matched values - need to identify correct column names
    ndci_col = 'ndci_cal' if 'ndci_cal' in matched.columns else 'ndci'
    chl_col = 'chl_ref' if 'chl_ref' in matched.columns else 'chl'
    
    X = matched[ndci_col].values.reshape(-1, 1)
    y = matched[chl_col].values
    
    # Remove any remaining NaN values
    mask = ~(np.isnan(X.flatten()) | np.isnan(y))
    X = X[mask].reshape(-1, 1)
    y = y[mask]
    
    if len(X) < 3:
        print("Not enough valid data points after removing NaN values")
        return None
    
    # Fit linear model
    linear_model = LinearRegression()
    linear_model.fit(X, y)
    y_pred_linear = linear_model.predict(X)
    r2_linear = r2_score(y, y_pred_linear)
    rmse_linear = np.sqrt(mean_squared_error(y, y_pred_linear))
    
    # Fit exponential model (if possible)
    try:
        # Initial guess for parameters
        p0 = [10, 1, 5]
        popt, _ = curve_fit(exponential_model, X.flatten(), y, p0=p0, maxfev=5000)
        y_pred_exp = exponential_model(X.flatten(), *popt)
        r2_exp = r2_score(y, y_pred_exp)
        rmse_exp = np.sqrt(mean_squared_error(y, y_pred_exp))
        exp_success = True
    except:
        exp_success = False
        r2_exp = 0
        rmse_exp = np.inf
        popt = [0, 0, 0]
    
    # Create calibration plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Scatter plot with regression lines
    ax1 = axes[0]
    ax1.scatter(X, y, alpha=0.5, label='Matched observations')
    
    # Plot linear fit
    X_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
    ax1.plot(X_range, linear_model.predict(X_range), 'r-', 
             label=f'Linear: y = {linear_model.coef_[0]:.1f}x + {linear_model.intercept_:.1f}\n'
                   f'R² = {r2_linear:.3f}, RMSE = {rmse_linear:.1f}')
    
    # Plot exponential fit if successful
    if exp_success:
        ax1.plot(X_range, exponential_model(X_range.flatten(), *popt), 'g-',
                label=f'Exponential: y = {popt[0]:.1f}*exp({popt[1]:.2f}x) + {popt[2]:.1f}\n'
                      f'R² = {r2_exp:.3f}, RMSE = {rmse_exp:.1f}')
    
    ax1.set_xlabel('NDCI')
    ax1.set_ylabel('MODIS Chlorophyll (µg/L)')
    ax1.set_title(f'{lake_name} - NDCI vs MODIS Calibration')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Residuals plot
    ax2 = axes[1]
    residuals = y - y_pred_linear
    ax2.scatter(y_pred_linear, residuals, alpha=0.5)
    ax2.axhline(y=0, color='r', linestyle='--')
    ax2.set_xlabel('Predicted Chlorophyll (µg/L)')
    ax2.set_ylabel('Residuals (µg/L)')
    ax2.set_title('Residual Analysis - Linear Model')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save calibration plot
    plot_filename = f"{lake_name}_calibration_plot.png"
    fig.savefig(plot_filename, dpi=300, bbox_inches='tight')
    print(f"Calibration plot saved as: {plot_filename}")
    
    # Return calibration results
    results = {
        'n_matches': len(matched),
        'linear_slope': linear_model.coef_[0],
        'linear_offset': linear_model.intercept_,
        'linear_r2': r2_linear,
        'linear_rmse': rmse_linear,
        'exp_success': exp_success,
        'exp_params': popt if exp_success else None,
        'exp_r2': r2_exp if exp_success else None,
        'exp_rmse': rmse_exp if exp_success else None,
        'matched_data': matched
    }
    
    return results

def print_calibration_summary(results, sensor_name):
    """
    Print a summary of calibration results.
    """
    print(f"\n{'='*60}")
    print(f"Calibration Results: {sensor_name}")
    print(f"{'='*60}")
    print(f"Matched observations: {results['n_matches']}")
    print(f"\nLinear Model: Chl = {results['linear_slope']:.1f} × NDCI + {results['linear_offset']:.1f}")
    print(f"  R² = {results['linear_r2']:.3f}")
    print(f"  RMSE = {results['linear_rmse']:.1f} µg/L")
    
    if results['exp_success']:
        a, b, c = results['exp_params']
        print(f"\nExponential Model: Chl = {a:.1f} × exp({b:.2f} × NDCI) + {c:.1f}")
        print(f"  R² = {results['exp_r2']:.3f}")
        print(f"  RMSE = {results['exp_rmse']:.1f} µg/L")
    else:
        print("\nExponential model fitting failed")
    
    # Recommendation
    if results['exp_success'] and results['exp_r2'] > results['linear_r2'] + 0.05:
        print("\n✓ Recommendation: Use exponential model (significantly better fit)")
    else:
        print("\n✓ Recommendation: Use linear model (simpler and adequate)")
    print(f"{'='*60}")

### Detroit Lake Calibration - Sentinel-2 vs MODIS

In [ ]:
"""
Calibrate Sentinel-2 NDCI to chlorophyll using MODIS as reference for Detroit Lake.
"""

# Load Sentinel-2 data
sentinel_detroit = read_data('Detroit_S2_NDCI_500m.csv')

# Ensure numeric data types
sentinel_detroit['ndci'] = pd.to_numeric(sentinel_detroit['ndci'], errors='coerce')
sentinel_detroit = sentinel_detroit.dropna(subset=['date', 'ndci'])

print(f"Sentinel-2 records: {len(sentinel_detroit)}")

# Perform calibration
print("\nCalibrating Sentinel-2 NDCI for Detroit Lake using MODIS reference...")
detroit_sentinel_cal = calibrate_ndci_to_chl(
    modis_combined,
    sentinel_detroit,
    lake_name="Detroit_Sentinel2"
)

# Print summary
if detroit_sentinel_cal:
    print_calibration_summary(detroit_sentinel_cal, "Detroit Lake - Sentinel-2")

### Apply Calibrated Coefficients - Detroit Lake

Now let's re-plot the data using the calibrated coefficients derived from the cross-sensor analysis.

In [ ]:
"""
Compare original vs calibrated chlorophyll estimates for Detroit Lake.
"""

# Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- Landsat Comparison ---
if detroit_landsat_cal:
    # Original coefficients
    df_orig = landsat_detroit.copy()
    df_orig['chl'] = df_orig['ndci'] * 30 + 3  # Original coefficients
    
    # Calibrated coefficients
    df_cal = landsat_detroit.copy()
    df_cal['chl'] = (df_cal['ndci'] * detroit_landsat_cal['linear_slope'] + 
                     detroit_landsat_cal['linear_offset'])
    
    # Plot Landsat
    ax = axes[0, 0]
    ax.plot(df_orig['date'], df_orig['chl'], 'b-', alpha=0.5, 
            label=f'Original (30×NDCI+3)')
    ax.plot(df_cal['date'], df_cal['chl'], 'r-', alpha=0.7,
            label=f"Calibrated ({detroit_landsat_cal['linear_slope']:.1f}×NDCI"
                  f"+{detroit_landsat_cal['linear_offset']:.1f})")
    ax.set_title('Detroit Lake - Landsat Chlorophyll')
    ax.set_ylabel('Chl-a (µg/L)')
    ax.legend()
    ax.grid(True, alpha=0.3)

# --- Sentinel-2 Comparison ---
if detroit_sentinel_cal:
    # Original coefficients
    df_orig = sentinel_detroit.copy()
    df_orig['chl'] = df_orig['ndci'] * 30 + 3  # Original coefficients
    
    # Calibrated coefficients
    df_cal = sentinel_detroit.copy()
    df_cal['chl'] = (df_cal['ndci'] * detroit_sentinel_cal['linear_slope'] + 
                     detroit_sentinel_cal['linear_offset'])
    
    # Plot Sentinel-2
    ax = axes[0, 1]
    ax.plot(df_orig['date'], df_orig['chl'], 'b-', alpha=0.5,
            label=f'Original (30×NDCI+3)')
    ax.plot(df_cal['date'], df_cal['chl'], 'r-', alpha=0.7,
            label=f"Calibrated ({detroit_sentinel_cal['linear_slope']:.1f}×NDCI"
                  f"+{detroit_sentinel_cal['linear_offset']:.1f})")
    ax.set_title('Detroit Lake - Sentinel-2 Chlorophyll')
    ax.set_ylabel('Chl-a (µg/L)')
    ax.legend()
    ax.grid(True, alpha=0.3)

# --- MODIS Reference ---
ax = axes[1, 0]
ax.plot(modis_aqua_clean['date'], modis_aqua_clean['chl'], 'g-', alpha=0.5,
        label='MODIS Aqua')
ax.plot(modis_terra_clean['date'], modis_terra_clean['chl'], 'orange', alpha=0.5,
        label='MODIS Terra')
ax.set_title('Detroit Lake - MODIS Reference Chlorophyll')
ax.set_ylabel('Chl-a (µg/L)')
ax.set_xlabel('Date')
ax.legend()
ax.grid(True, alpha=0.3)

# --- All sensors combined (calibrated) ---
ax = axes[1, 1]
if detroit_landsat_cal:
    df_cal = landsat_detroit.copy()
    df_cal['chl'] = (df_cal['ndci'] * detroit_landsat_cal['linear_slope'] + 
                     detroit_landsat_cal['linear_offset'])
    ax.plot(df_cal['date'], df_cal['chl'], 'b.', alpha=0.3, markersize=3,
            label='Landsat (calibrated)')

if detroit_sentinel_cal:
    df_cal = sentinel_detroit.copy()
    df_cal['chl'] = (df_cal['ndci'] * detroit_sentinel_cal['linear_slope'] + 
                     detroit_sentinel_cal['linear_offset'])
    ax.plot(df_cal['date'], df_cal['chl'], 'r.', alpha=0.3, markersize=3,
            label='Sentinel-2 (calibrated)')

ax.plot(modis_combined['date'], modis_combined['chl'], 'g.', alpha=0.3, markersize=2,
        label='MODIS (reference)')

ax.set_title('Detroit Lake - All Sensors Comparison (Calibrated)')
ax.set_ylabel('Chl-a (µg/L)')
ax.set_xlabel('Date')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()

# Save comparison plot
comparison_filename = "Detroit_Lake_calibration_comparison.png"
fig.savefig(comparison_filename, dpi=300, bbox_inches='tight')
print(f"\nComparison plot saved as: {comparison_filename}")

### Klamath Lake Calibration - Sentinel-2 vs. MODIS

In [ ]:
"""
Calibrate Sentinel-2 NDCI to chlorophyll using MODIS as reference for Klamath Lake.
"""

# Load Sentinel-2 data
sentinel_Klamath = read_data('Klamath_S2_NDCI_500m.csv')

# Ensure numeric data types
sentinel_Klamath['ndci'] = pd.to_numeric(sentinel_Klamath['ndci'], errors='coerce')
sentinel_Klamath = sentinel_Klamath.dropna(subset=['date', 'ndci'])

print(f"Sentinel-2 records: {len(sentinel_Klamath)}")

# Perform calibration
print("\nCalibrating Sentinel-2 NDCI for Klamath Lake using MODIS reference...")
Klamath_sentinel_cal = calibrate_ndci_to_chl(
    modis_combined,
    sentinel_Klamath,
    lake_name="Klamath_Sentinel2"
)

# Print summary
if Klamath_sentinel_cal:
    print_calibration_summary(Klamath_sentinel_cal, "Klamath Lake - Sentinel-2")

### Apply Calibrated Coefficients - Klamath Lake

In [ ]:
"""
Compare original vs calibrated chlorophyll estimates for Klamath Lake.
"""

# Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- Landsat Comparison ---
if Klamath_landsat_cal:
    # Original coefficients
    df_orig = landsat_Klamath.copy()
    df_orig['chl'] = df_orig['ndci'] * 30 + 3  # Original coefficients
    
    # Calibrated coefficients
    df_cal = landsat_Klamath.copy()
    df_cal['chl'] = (df_cal['ndci'] * Klamath_landsat_cal['linear_slope'] + 
                     Klamath_landsat_cal['linear_offset'])
    
    # Plot Landsat
    ax = axes[0, 0]
    ax.plot(df_orig['date'], df_orig['chl'], 'b-', alpha=0.5, 
            label=f'Original (30×NDCI+3)')
    ax.plot(df_cal['date'], df_cal['chl'], 'r-', alpha=0.7,
            label=f"Calibrated ({Klamath_landsat_cal['linear_slope']:.1f}×NDCI"
                  f"+{Klamath_landsat_cal['linear_offset']:.1f})")
    ax.set_title('Klamath Lake - Landsat Chlorophyll')
    ax.set_ylabel('Chl-a (µg/L)')
    ax.legend()
    ax.grid(True, alpha=0.3)

# --- Sentinel-2 Comparison ---
if Klamath_sentinel_cal:
    # Original coefficients
    df_orig = sentinel_Klamath.copy()
    df_orig['chl'] = df_orig['ndci'] * 30 + 3  # Original coefficients
    
    # Calibrated coefficients
    df_cal = sentinel_Klamath.copy()
    df_cal['chl'] = (df_cal['ndci'] * Klamath_sentinel_cal['linear_slope'] + 
                     Klamath_sentinel_cal['linear_offset'])
    
    # Plot Sentinel-2
    ax = axes[0, 1]
    ax.plot(df_orig['date'], df_orig['chl'], 'b-', alpha=0.5,
            label=f'Original (30×NDCI+3)')
    ax.plot(df_cal['date'], df_cal['chl'], 'r-', alpha=0.7,
            label=f"Calibrated ({Klamath_sentinel_cal['linear_slope']:.1f}×NDCI"
                  f"+{Klamath_sentinel_cal['linear_offset']:.1f})")
    ax.set_title('Klamath Lake - Sentinel-2 Chlorophyll')
    ax.set_ylabel('Chl-a (µg/L)')
    ax.legend()
    ax.grid(True, alpha=0.3)

# --- MODIS Reference ---
ax = axes[1, 0]
ax.plot(modis_aqua_clean['date'], modis_aqua_clean['chl'], 'g-', alpha=0.5,
        label='MODIS Aqua')
ax.plot(modis_terra_clean['date'], modis_terra_clean['chl'], 'orange', alpha=0.5,
        label='MODIS Terra')
ax.set_title('Klamath Lake - MODIS Reference Chlorophyll')
ax.set_ylabel('Chl-a (µg/L)')
ax.set_xlabel('Date')
ax.legend()
ax.grid(True, alpha=0.3)

# --- All sensors combined (calibrated) ---
ax = axes[1, 1]
if Klamath_landsat_cal:
    df_cal = landsat_Klamath.copy()
    df_cal['chl'] = (df_cal['ndci'] * Klamath_landsat_cal['linear_slope'] + 
                     Klamath_landsat_cal['linear_offset'])
    ax.plot(df_cal['date'], df_cal['chl'], 'b.', alpha=0.3, markersize=3,
            label='Landsat (calibrated)')

if Klamath_sentinel_cal:
    df_cal = sentinel_Klamath.copy()
    df_cal['chl'] = (df_cal['ndci'] * Klamath_sentinel_cal['linear_slope'] + 
                     Klamath_sentinel_cal['linear_offset'])
    ax.plot(df_cal['date'], df_cal['chl'], 'r.', alpha=0.3, markersize=3,
            label='Sentinel-2 (calibrated)')

ax.plot(modis_combined['date'], modis_combined['chl'], 'g.', alpha=0.3, markersize=2,
        label='MODIS (reference)')

ax.set_title('Klamath Lake - All Sensors Comparison (Calibrated)')
ax.set_ylabel('Chl-a (µg/L)')
ax.set_xlabel('Date')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()

# Save comparison plot
comparison_filename = "Detroit_Lake_calibration_comparison.png"
fig.savefig(comparison_filename, dpi=300, bbox_inches='tight')
print(f"\nComparison plot saved as: {comparison_filename}")